In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import cv2
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from torch import Tensor
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam

class BottleDataset(Dataset):
    def __init__(self, image_paths: list, labels: list) -> None:
        self.image_paths = image_paths
        self.labels = labels

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        path = str(self.image_paths[idx])
        # Carrega a imagem pré-processada em cinza
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        
        # Normalização dos pixels para o intervalo [0, 1] e adição do canal (1, H, W)
        img_tensor = torch.from_numpy(img).float().unsqueeze(0) / 255.0
        label_tensor = torch.tensor(self.labels[idx]).long()
        
        return img_tensor, label_tensor

In [3]:
# Mapeia as imagens já redimensionadas
garrafa_dir = Path("./Dados/Redimensionados/Garrafa")
bg_dir = Path("./Dados/Redimensionados/Background")

garrafa_paths = list(garrafa_dir.glob("*.png"))
bg_paths = list(bg_dir.glob("*.png"))

# Define os rótulos: 1 para Garrafa, 0 para Background
all_paths = np.array(garrafa_paths + bg_paths)
all_labels = np.array([1] * len(garrafa_paths) + [0] * len(bg_paths))

print(f"Total de Garrafas: {len(garrafa_paths)}")
print(f"Total de Backgrounds: {len(bg_paths)}")
print(f"Total Geral: {len(all_paths)}")

Total de Garrafas: 645
Total de Backgrounds: 805
Total Geral: 1450


In [4]:
class BottleCNN(nn.Module):
    def __init__(self):
        super(BottleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),  # Entrada: 1 canal (cinza)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # Saída: 16 x 80 x 60
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)                           # Saída: 32 x 40 x 30
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 40 * 30, 64),
            nn.ReLU(),
            nn.Linear(64, 2)                            # 2 classes: Background (0) e Garrafa (1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [7]:
def train_kfold_images(paths, labels, k_folds=5, epochs=15, batch_size=16, lr=0.001):
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    acc_scores = []
    f1_scores = []

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Utilizando dispositivo: {device}\n")

    for fold, (train_idx, test_idx) in enumerate(kf.split(paths)):
        print(f"--- Fold {fold + 1}/{k_folds} ---")

        # Garante a re-inicialização do modelo em cada fold
        model = BottleCNN().to(device)
        optimizer = Adam(model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()

        train_data = BottleDataset(paths[train_idx], labels[train_idx])
        test_data = BottleDataset(paths[test_idx], labels[test_idx])

        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

        # Treinamento do Fold
        for epoch in range(epochs):
            model.train()
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                out = model(x)
                loss = criterion(out, y)
                loss.backward()
                optimizer.step()

        # Avaliação do Fold
        model.eval()
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for x, y in test_loader:
                x = x.to(device)
                out = model(x)
                preds = torch.argmax(out, dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_targets.extend(y.numpy())

        acc = accuracy_score(all_targets, all_preds)
        f1 = f1_score(all_targets, all_preds, average="binary")
        cm = confusion_matrix(all_targets, all_preds)

        acc_scores.append(acc)
        f1_scores.append(f1)

        print(f"Acurácia Teste Fold {fold + 1}: {acc * 100:.2f}% | F1-Score: {f1:.4f}")
        print(f"Matriz de Confusão:\n{cm}\n")

    print("==========================================")
    print(f"Resultado Médio Final ----- Acurácia: {np.mean(acc_scores)*100:.2f}% | F1-Score: {np.mean(f1_scores):.4f}")
    print("==========================================")

# Executar o treino
train_kfold_images(all_paths, all_labels, k_folds=5, epochs=30, batch_size=16, lr=0.001)

Utilizando dispositivo: cuda

--- Fold 1/5 ---
Acurácia Teste Fold 1: 70.00% | F1-Score: 0.6717
Matriz de Confusão:
[[114  42]
 [ 45  89]]

--- Fold 2/5 ---
Acurácia Teste Fold 2: 72.76% | F1-Score: 0.6749
Matriz de Confusão:
[[129  32]
 [ 47  82]]

--- Fold 3/5 ---
Acurácia Teste Fold 3: 67.93% | F1-Score: 0.6353
Matriz de Confusão:
[[116  41]
 [ 52  81]]

--- Fold 4/5 ---
Acurácia Teste Fold 4: 67.93% | F1-Score: 0.6235
Matriz de Confusão:
[[120  37]
 [ 56  77]]

--- Fold 5/5 ---
Acurácia Teste Fold 5: 67.93% | F1-Score: 0.5974
Matriz de Confusão:
[[128  46]
 [ 47  69]]

Resultado Médio Final ----- Acurácia: 69.31% | F1-Score: 0.6406


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam

def train_stratified_balanced_kfold(paths, labels, k_folds=5, epochs=15, batch_size=16, lr=0.001):
    # Usando StratifiedKFold para manter a proporção das classes
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
    
    acc_scores = []
    f1_scores = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Passamos paths e labels no skf.split
    for fold, (train_idx, test_idx) in enumerate(skf.split(paths, labels)):
        print(f"--- Fold {fold + 1}/{k_folds} ---")

        # 1. Cálculo dos Pesos das Classes no Treino do Fold Atual
        train_labels_fold = labels[train_idx]
        class_counts = np.bincount(train_labels_fold)
        
        # Evita divisão por zero se alguma classe não existir
        class_weights = len(train_labels_fold) / (2.0 * class_counts)
        class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

        print(f"Contagem [Background, Garrafa]: {class_counts}")
        print(f"Pesos aplicados [Background, Garrafa]: {class_weights.round(3)}")

        # 2. Inicialização do Modelo, Perda Ponderada e Otimizador
        model = BottleCNN().to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
        optimizer = Adam(model.parameters(), lr=lr)

        # 3. Preparação dos DataLoaders
        train_data = BottleDataset(paths[train_idx], labels[train_idx])
        test_data = BottleDataset(paths[test_idx], labels[test_idx])

        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

        # 4. Treinamento
        for epoch in range(epochs):
            model.train()
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                out = model(x)
                loss = criterion(out, y)
                loss.backward()
                optimizer.step()

        # 5. Avaliação do Fold
        model.eval()
        all_preds, all_targets = [], []

        with torch.no_grad():
            for x, y in test_loader:
                x = x.to(device)
                out = model(x)
                preds = torch.argmax(out, dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_targets.extend(y.numpy())

        acc = accuracy_score(all_targets, all_preds)
        f1 = f1_score(all_targets, all_preds, average="binary")
        cm = confusion_matrix(all_targets, all_preds)

        acc_scores.append(acc)
        f1_scores.append(f1)

        print(f"Acurácia Teste: {acc * 100:.2f}% | F1-Score: {f1:.4f}")
        print(f"Matriz de Confusão:\n{cm}\n")

    print("==========================================")
    print(f"Resultado Médio Final ----- Acurácia: {np.mean(acc_scores)*100:.2f}% | F1-Score: {np.mean(f1_scores):.4f}")
    print("==========================================")

# Executar treinamento balanceado
train_stratified_balanced_kfold(all_paths, all_labels, k_folds=5, epochs=15)

--- Fold 1/5 ---
Contagem [Background, Garrafa]: [644 516]
Pesos aplicados [Background, Garrafa]: [0.901 1.124]
Acurácia Teste: 68.62% | F1-Score: 0.6061
Matriz de Confusão:
[[129  32]
 [ 59  70]]

--- Fold 2/5 ---
Contagem [Background, Garrafa]: [644 516]
Pesos aplicados [Background, Garrafa]: [0.901 1.124]
Acurácia Teste: 64.83% | F1-Score: 0.5750
Matriz de Confusão:
[[119  42]
 [ 60  69]]

--- Fold 3/5 ---
Contagem [Background, Garrafa]: [644 516]
Pesos aplicados [Background, Garrafa]: [0.901 1.124]
Acurácia Teste: 65.17% | F1-Score: 0.6189
Matriz de Confusão:
[[107  54]
 [ 47  82]]

--- Fold 4/5 ---
Contagem [Background, Garrafa]: [644 516]
Pesos aplicados [Background, Garrafa]: [0.901 1.124]
Acurácia Teste: 67.59% | F1-Score: 0.6594
Matriz de Confusão:
[[105  56]
 [ 38  91]]

--- Fold 5/5 ---
Contagem [Background, Garrafa]: [644 516]
Pesos aplicados [Background, Garrafa]: [0.901 1.124]
Acurácia Teste: 71.38% | F1-Score: 0.6640
Matriz de Confusão:
[[125  36]
 [ 47  82]]

Resultado 